# 08 — Pipeline effect decomposition

Quantifies which analytical decisions most strongly influence cross-pipeline agreement using pairwise contrasts and a secondary ANOVA decomposition.


In [1]:
import os, sys, json, warnings, hashlib, platform, time, math, itertools, shutil
from pathlib import Path
os.environ.setdefault('OMP_NUM_THREADS','2'); os.environ.setdefault('OPENBLAS_NUM_THREADS','2'); os.environ.setdefault('MKL_NUM_THREADS','2')
warnings.filterwarnings('ignore')
def find_project_root(start=None):
    start=Path(start or os.getcwd()).resolve()
    for p in [start]+list(start.parents):
        if (p/'config.json').exists() and (p/'notebooks').exists() and (p/'data').exists(): return p
    for base in [Path('/content'),Path('/mnt/data')]:
        if base.exists():
            hits=list(base.rglob('PCOS_Phenotype_Robustness_Framework_v*_colab/config.json'))
            if hits: return hits[0].parent
    raise FileNotFoundError('Project root not found')
PROJECT_ROOT=find_project_root(); os.chdir(PROJECT_ROOT); print('PROJECT_ROOT=',PROJECT_ROOT)
# Colab/bootstrap dependencies. The notebooks are self-contained and do not import project .py files.
# In Google Colab this cell installs requirements automatically when imports are missing.
# Set PCOS_AUTO_INSTALL=0 to disable installation, or PCOS_AUTO_INSTALL=1 to force it.
def _maybe_install_requirements():
    import importlib.util, subprocess
    required = ['numpy','pandas','sklearn','matplotlib','scipy','openpyxl']
    missing = [m for m in required if importlib.util.find_spec(m) is None]
    force = os.environ.get('PCOS_AUTO_INSTALL','auto') == '1'
    disable = os.environ.get('PCOS_AUTO_INSTALL','auto') == '0'
    if (missing or force) and not disable:
        req = PROJECT_ROOT/'requirements.txt'
        if req.exists():
            print('Installing requirements because missing packages were detected:', missing)
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)])
_maybe_install_requirements()
CONFIG=json.load(open('config.json','r',encoding='utf-8'))
ACTIVE_MODE=os.environ.get('PCOS_PIPELINE_MODE',CONFIG.get('active_mode','PILOT_REVIEW_FAST'))
MODE=CONFIG['execution_modes'][ACTIVE_MODE]
print('ACTIVE_MODE=',ACTIVE_MODE, MODE)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import BayesianRidge
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer
from sklearn.decomposition import PCA, FastICA
from sklearn.cluster import KMeans, AgglomerativeClustering, SpectralClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, silhouette_score
ROOT=Path('.'); OUT=ROOT/'outputs'; DATA=ROOT/'data'; OUT.mkdir(exist_ok=True); DATA.mkdir(exist_ok=True)
MIN_CLUSTER_FRAC=float(CONFIG.get('min_cluster_fraction',0.05))
def ensure_dir(p): p=Path(p); p.mkdir(parents=True,exist_ok=True); return p
def save_df(df,path):
    path=Path(path); ensure_dir(path.parent)
    if path.suffix=='.parquet':
        try:
            df.to_parquet(path,index=False)
        except Exception as e:
            alt=path.with_suffix('.csv')
            df.to_csv(alt,index=False)
            print(f'[parquet fallback] Could not write {path.name} ({type(e).__name__}); wrote {alt.name} instead.')
    elif path.suffix=='.xlsx':
        df.to_excel(path,index=False)
    else:
        df.to_csv(path,index=False)
def read_df(path):
    path=Path(path)
    if path.suffix=='.parquet':
        if path.exists():
            try:
                return pd.read_parquet(path)
            except Exception as e:
                alt=path.with_suffix('.csv')
                if alt.exists():
                    print(f'[parquet fallback] Could not read {path.name} ({type(e).__name__}); reading {alt.name}.')
                    return pd.read_csv(alt)
                raise
        alt=path.with_suffix('.csv')
        if alt.exists():
            return pd.read_csv(alt)
        raise FileNotFoundError(f'Neither {path} nor fallback {alt} exists')
    if path.suffix in ['.xlsx','.xls']:
        return pd.read_excel(path)
    return pd.read_csv(path)
def numeric_clean(s):
    if pd.api.types.is_numeric_dtype(s): return pd.to_numeric(s, errors='coerce')
    x=s.astype(str).str.replace(',', '.', regex=False).str.replace(r'[^0-9eE+\-\.<>=]', '', regex=True).str.replace(r'^[<>]=?', '', regex=True).replace({'':np.nan,'nan':np.nan,'None':np.nan})
    return pd.to_numeric(x, errors='coerce')
def coalesce_numeric(df, candidates):
    cols=[c for c in candidates if c in df.columns]
    if not cols: return pd.Series(np.nan,index=df.index,dtype=float), None
    out=pd.Series(np.nan,index=df.index,dtype=float); src=[]
    for c in cols:
        v=numeric_clean(df[c]); mask=out.isna()&v.notna(); out[mask]=v[mask]
        if v.notna().sum()>0: src.append(f'{c} (n={int(v.notna().sum())})')
    return out, '; '.join(src[:4])
def cluster_valid(labels,n):
    vals,counts=np.unique(labels,return_counts=True)
    if len(vals)<2: return False,0.0,len(vals)
    mf=counts.min()/n
    return bool(mf>=MIN_CLUSTER_FRAC),float(mf),len(vals)
def fit_cluster(X,method='gmm',seed=0,k=2):
    n=len(X)
    if method=='kmeans': labels=KMeans(n_clusters=k,n_init=20,random_state=seed).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='gmm':
        m=GaussianMixture(n_components=k,covariance_type='diag',reg_covar=1e-6,n_init=3,max_iter=100,random_state=seed).fit(X); labels=m.predict(X); unc=1-m.predict_proba(X).max(axis=1)
    elif method=='hierarchical': labels=AgglomerativeClustering(n_clusters=k,linkage='ward').fit_predict(X); unc=np.full(n,np.nan)
    elif method=='spectral': labels=SpectralClustering(n_clusters=k,assign_labels='kmeans',random_state=seed,affinity='nearest_neighbors',n_neighbors=min(15,max(2,n//20))).fit_predict(X); unc=np.full(n,np.nan)
    elif method=='consensus':
        # Fast consensus-by-medoids for Colab: generate an ensemble of k-means partitions
        # and choose the partition with the highest mean ARI to all other ensemble members.
        labs=[KMeans(n_clusters=k,n_init=5,random_state=seed+i).fit_predict(X) for i in range(8)]
        S=np.zeros((len(labs),len(labs)))
        for i in range(len(labs)):
            for j in range(i+1,len(labs)):
                S[i,j]=S[j,i]=adjusted_rand_score(labs[i],labs[j])
        labels=labs[int(np.argmax(S.mean(axis=1)))]
        unc=np.full(n, max(0.0, 1.0-float(S.mean())))
    else: raise ValueError(method)
    return np.asarray(labels).astype(int),unc
def transform_matrix(Xdf,imputation,scaling,reduction,seed=0):
    X=Xdf.copy()
    if imputation=='complete_case':
        keep=~X.isna().any(axis=1); X=X.loc[keep].copy(); ids=X.index.to_numpy(); arr=X.to_numpy(float)
    else:
        ids=X.index.to_numpy(); arr=X.to_numpy(float)
        if imputation=='median':
            arr=SimpleImputer(strategy='median').fit_transform(arr)
        elif imputation=='knn':
            arr=KNNImputer(n_neighbors=5).fit_transform(arr)
        elif imputation=='mice':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    arr=IterativeImputer(estimator=BayesianRidge(), max_iter=4, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible deterministic MICE proxy used for the frozen manuscript run.
                # Optional true iterative imputation can be enabled with PCOS_TRUE_ITERATIVE_IMPUTERS=1.
                arr=base
        elif imputation=='missforest_like':
            base=SimpleImputer(strategy='median').fit_transform(arr)
            if (ACTIVE_MODE in ['MANUSCRIPT_FINAL','PUBLICATION_FULL','PUBLICATION_EXHAUSTIVE']) or os.environ.get('PCOS_TRUE_ITERATIVE_IMPUTERS','0')=='1':
                try:
                    est=ExtraTreesRegressor(n_estimators=10, random_state=seed, n_jobs=1, max_depth=6)
                    arr=IterativeImputer(estimator=est, max_iter=2, random_state=seed, initial_strategy='median', skip_complete=True).fit_transform(arr)
                except Exception:
                    arr=base
            else:
                # Colab-feasible MissForest-like proxy; KNN preserves local multivariate structure without very long runtime.
                try:
                    arr=KNNImputer(n_neighbors=7).fit_transform(arr)
                except Exception:
                    arr=base
        else: raise ValueError(imputation)
    if scaling=='zscore': arr=StandardScaler().fit_transform(arr)
    elif scaling=='robust': arr=RobustScaler().fit_transform(arr)
    elif scaling=='quantile': arr=QuantileTransformer(output_distribution='normal',n_quantiles=min(200,len(arr)),random_state=seed).fit_transform(arr)
    else: raise ValueError(scaling)
    if reduction=='none': Z=arr
    elif reduction.startswith('pca'): Z=PCA(n_components=float(reduction.replace('pca',''))/100,svd_solver='full',random_state=seed).fit_transform(arr)
    elif reduction=='ica': Z=FastICA(n_components=min(10,arr.shape[1],len(arr)-1),random_state=seed,max_iter=500,whiten='unit-variance').fit_transform(arr)
    else: raise ValueError(reduction)
    return Z,ids
def pairwise_coassignment(labels,ids,all_ids):
    M=np.full((len(all_ids),len(all_ids)),np.nan); loc={v:i for i,v in enumerate(all_ids)}; idx=[loc[i] for i in ids]
    C=(labels[:,None]==labels[None,:]).astype(float); M[np.ix_(idx,idx)]=C; return M


def variation_of_information(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n==0: return np.nan
    _,ai=np.unique(a,return_inverse=True); _,bi=np.unique(b,return_inverse=True)
    pa=np.bincount(ai)/n; pb=np.bincount(bi)/n
    H_a=-(pa*np.log2(pa+1e-12)).sum(); H_b=-(pb*np.log2(pb+1e-12)).sum()
    I=0.0
    for i in range(len(pa)):
        for j in range(len(pb)):
            pij=np.mean((ai==i)&(bi==j))
            if pij>0: I += pij*np.log2(pij/(pa[i]*pb[j])+1e-12)
    return float(H_a+H_b-2*I)

def coassignment_jaccard(a,b):
    a=np.asarray(a); b=np.asarray(b); n=len(a)
    if n<3: return np.nan
    A=(a[:,None]==a[None,:]); B=(b[:,None]==b[None,:])
    tri=np.triu_indices(n,1)
    A=A[tri]; B=B[tri]
    union=np.logical_or(A,B).sum()
    if union==0: return np.nan
    return float(np.logical_and(A,B).sum()/union)

def bootstrap_ci(x, n_boot=200, seed=42, q=(2.5,97.5)):
    x=np.asarray(pd.Series(x).dropna(),float)
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng(seed); vals=[]
    for _ in range(int(n_boot)):
        vals.append(np.median(rng.choice(x, size=len(x), replace=True)))
    return tuple(np.percentile(vals,q))


PROJECT_ROOT= /mnt/data/user_dataset_run/PCOS_Phenotype_Robustness_Framework_v4_final_colab
ACTIVE_MODE= PILOT_REVIEW_FAST {'seeds': [0, 1, 2], 'max_specs': 36, 'bootstrap_n': 75}


In [ ]:
out=ensure_dir(OUT/'08_pipeline_effects')
pair=read_df(OUT/'07_pairwise_robustness/all_pairwise_partition_agreement.csv')
valid=read_df(OUT/'06_pipeline_perturbation/pipeline_perturbation_results_valid_only.csv')
# Same-versus-different factor effect on pairwise ARI, with bootstrap confidence intervals.
rows=[]; rng=np.random.default_rng(CONFIG['random_seed']); B=MODE.get('bootstrap_n',200)
for f in ['imputation','scaling','reduction','clustering','seed']:
    same=pair.loc[pair[f'same_{f}'],'ari'].dropna().values; diff=pair.loc[~pair[f'same_{f}'],'ari'].dropna().values
    effect=float(np.median(same)-np.median(diff)) if len(same) and len(diff) else np.nan
    boots=[]
    for _ in range(B):
        if len(same) and len(diff): boots.append(np.median(rng.choice(same,len(same),replace=True))-np.median(rng.choice(diff,len(diff),replace=True)))
    lo,hi=(np.quantile(boots,[.025,.975]) if boots else (np.nan,np.nan))
    rows.append({'factor':f,'n_same_pairs':len(same),'n_different_pairs':len(diff),'median_ari_same':np.median(same) if len(same) else np.nan,'median_ari_different':np.median(diff) if len(diff) else np.nan,'same_factor_ari_advantage':effect,'bootstrap_ci_low':lo,'bootstrap_ci_high':hi})
eff=pd.DataFrame(rows).sort_values('same_factor_ari_advantage',ascending=False); save_df(eff,out/'pairwise_factor_effects.csv')
# OLS partial eta-squared on reference-relative ARI (secondary decomposition; descriptive, not causal).
import statsmodels.api as sm
import statsmodels.formula.api as smf
formula='ari_vs_reference ~ C(imputation)+C(scaling)+C(reduction)+C(clustering)+C(seed)'
model=smf.ols(formula,data=valid.dropna(subset=['ari_vs_reference'])).fit()
anova=sm.stats.anova_lm(model,typ=2).reset_index().rename(columns={'index':'term'})
resid=float(anova.loc[anova.term=='Residual','sum_sq'].iloc[0])
anova['partial_eta_squared']=anova.apply(lambda r: r['sum_sq']/(r['sum_sq']+resid) if r['term']!='Residual' else np.nan,axis=1)
save_df(anova,out/'reference_relative_anova_effects.csv')
with open(out/'reference_relative_ols_summary.txt','w') as fh: fh.write(model.summary().as_text())
plt.figure(figsize=(7,4)); z=eff.sort_values('same_factor_ari_advantage'); plt.barh(z.factor,z.same_factor_ari_advantage,xerr=[z.same_factor_ari_advantage-z.bootstrap_ci_low,z.bootstrap_ci_high-z.same_factor_ari_advantage]); plt.xlabel('Median pairwise ARI advantage when factor is shared'); plt.tight_layout(); plt.savefig(out/'Figure4_pipeline_factor_effects.png',dpi=300); plt.close()
print(eff.to_string(index=False)); print(anova[['term','partial_eta_squared']].to_string(index=False))


# Formal, reference-free distance-based variance decomposition.
# The response is the all-by-all partition dissimilarity matrix D = sqrt(max(0, 1 - ARI)).
# Marginal effects are estimated by comparing full and factor-reduced projection models.
from numpy.linalg import matrix_rank, pinv

valid_meta=read_df(OUT/'06_pipeline_perturbation/pipeline_perturbation_results_valid_only.csv').copy()
ari_mat=read_df(OUT/'07_pairwise_robustness/pairwise_ari_matrix.csv')
# The first column may contain row labels after CSV export.
if ari_mat.columns[0] not in [str(x) for x in valid_meta.run_id.astype(int).tolist()]:
    ari_mat=ari_mat.set_index(ari_mat.columns[0])
else:
    ari_mat.index=ari_mat.columns
ari_mat.index=ari_mat.index.astype(str); ari_mat.columns=ari_mat.columns.astype(str)
run_ids=[str(int(x)) for x in valid_meta.run_id]
ari_mat=ari_mat.loc[run_ids,run_ids].astype(float)
Ari=ari_mat.to_numpy()
np.fill_diagonal(Ari,1.0)
D=np.sqrt(np.clip(1.0-Ari,0,None))
n=D.shape[0]
Hc=np.eye(n)-np.ones((n,n))/n
G=-0.5*Hc@(D**2)@Hc
factors=['imputation','scaling','reduction','clustering','seed']

def design_matrix(df, include):
    parts=[np.ones((len(df),1))]
    for f in include:
        d=pd.get_dummies(df[f].astype(str),prefix=f,drop_first=True,dtype=float)
        if d.shape[1]: parts.append(d.to_numpy(float))
    return np.column_stack(parts)

def hat(X): return X@pinv(X.T@X)@X.T

def marginal_db_effect(meta,G,factor):
    Xf=design_matrix(meta,factors); Xr=design_matrix(meta,[f for f in factors if f!=factor])
    Pf=hat(Xf); Pr=hat(Xr)
    df1=max(1,matrix_rank(Xf)-matrix_rank(Xr)); df2=max(1,len(meta)-matrix_rank(Xf))
    ss=float(np.trace((Pf-Pr)@G)); ss_res=float(np.trace((np.eye(len(meta))-Pf)@G))
    ss=max(ss,0.0); ss_res=max(ss_res,1e-12)
    F=(ss/df1)/(ss_res/df2)
    r2=ss/(ss+ss_res)
    return ss,ss_res,df1,df2,F,r2

B=int(MODE.get('bootstrap_n',200)); rng=np.random.default_rng(CONFIG['random_seed'])
perm_B=min(max(B,199),999)
formal=[]
for factor in factors:
    ss,ssr,df1,df2,Fobs,r2=marginal_db_effect(valid_meta,G,factor)
    # Conditional label permutation: permute the tested factor while retaining all other factors.
    Fperm=[]
    for _ in range(perm_B):
        mp=valid_meta.copy(); mp[factor]=rng.permutation(mp[factor].to_numpy())
        try: Fperm.append(marginal_db_effect(mp,G,factor)[4])
        except Exception: pass
    p_perm=(1+np.sum(np.asarray(Fperm)>=Fobs))/(1+len(Fperm)) if Fperm else np.nan
    # Pipeline bootstrap for marginal R2. Duplicate pipelines are allowed; singular designs use pseudoinverse.
    rb=[]
    for _ in range(B):
        idx=rng.integers(0,n,n)
        mb=valid_meta.iloc[idx].reset_index(drop=True); Gb=G[np.ix_(idx,idx)]
        try:
            val=marginal_db_effect(mb,Gb,factor)[5]
            if np.isfinite(val): rb.append(val)
        except Exception: pass
    formal.append({'factor':factor,'marginal_sum_of_squares':ss,'residual_sum_of_squares':ssr,
                   'df_factor':df1,'df_residual':df2,'pseudo_F':Fobs,'marginal_partial_R2':r2,
                   'bootstrap_R2_ci_low':np.quantile(rb,.025) if rb else np.nan,
                   'bootstrap_R2_ci_high':np.quantile(rb,.975) if rb else np.nan,
                   'permutation_p_value':p_perm,'n_permutations':len(Fperm),'n_bootstrap':len(rb)})
formal=pd.DataFrame(formal).sort_values('marginal_partial_R2',ascending=False)
formal['relative_marginal_contribution_percent']=100*formal.marginal_partial_R2/formal.marginal_partial_R2.sum()
save_df(formal,out/'formal_distance_based_variance_decomposition.csv')
save_df(formal[['factor','marginal_partial_R2','bootstrap_R2_ci_low','bootstrap_R2_ci_high','relative_marginal_contribution_percent','pseudo_F','permutation_p_value']],out/'variance_contribution_summary.csv')
plt.figure(figsize=(7,4.5)); z=formal.sort_values('marginal_partial_R2')
err=np.vstack([np.maximum(0,z.marginal_partial_R2-z.bootstrap_R2_ci_low),np.maximum(0,z.bootstrap_R2_ci_high-z.marginal_partial_R2)])
plt.barh(z.factor,z.marginal_partial_R2,xerr=err)
plt.xlabel('Marginal partial $R^2$ for partition dissimilarity')
plt.tight_layout(); plt.savefig(out/'Figure4b_formal_variance_decomposition.png',dpi=300); plt.close()
print('\nFormal distance-based variance decomposition:')
print(formal[['factor','marginal_partial_R2','bootstrap_R2_ci_low','bootstrap_R2_ci_high','relative_marginal_contribution_percent','permutation_p_value']].to_string(index=False))
